# Laptop null bundle — summary

**Task** `laptop-null-bundle`. Details, commands and caveats: `verification.md`;
decisions and measured corrections: `decisions.md`; claim-by-claim check: `audit.md`.

## What this PR changes

The Streamlit app no longer loads any DWPC matrix. It used to preload all 52
Gene -> Biological Process matrices, 33.6 GB in memory, which does not fit a
24 GB laptop. Now:

- **Alpine** precomputes, for every metapath x target, a summary of each
  capacity stratum (size, score mean, centered sum of squares, capacity range):
  the null bundle, 97.8M rows, 3.07 GB on disk.
- **The laptop** computes the query genes' DWPC from the 3.3 MB edge files, reads
  the target's stratum summaries, removes the query genes from their strata and
  computes the same exact null moments, z and p.

Hypothesis (design.md): the laptop runs the full query in <= 1.5 GB, in <= 1 s
for 18 genes and <= 3 s for 200 genes, with z matching the matrix-based adapter.
The tables and figures below are read from this folder.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, display

BASE = Path.cwd()
if not (BASE / "tables").exists():
    BASE = Path("docs/tasks/laptop-null-bundle")
TABLES = BASE / "tables"
FIGURES = BASE / "figures"

## Agreement with the matrix-based adapter (Alpine validation)

Every metapath, 496 targets, random gene sets of 5-500 genes and annotated gene
sets (360 targets), scored both ways. Criteria (design.md): 0 stratum mismatches,
max z relative difference <= 1e-9, and NaN agreement with rows inside the
zero-variance band tabled. The negative control gives the query genes the
capacities of random other genes (wrong strata); it must disagree.

In [ ]:
display(json.loads((TABLES / "validate_verdict.json").read_text()))

In [ ]:
Image(filename=str(FIGURES / "validate_z_scatter.png"))

In [ ]:
Image(filename=str(FIGURES / "validate_z_rel_diff_hist.png"))

**Result: the numeric criteria pass; NaN agreement does not, for 23 rows (next
table).** 0 stratum mismatches, max z relative difference 7.1e-10, merges
identical on every row, and the shuffled control departs (median |dz| 1.46).
The laptop path sits on the diagonal; the control fills the plane.

In [ ]:
by_metapath = pd.read_csv(TABLES / "validate_by_metapath.csv")
display(by_metapath.sort_values("max_z_rel_diff", ascending=False).head(10))

In [ ]:
nan_rows = pd.read_csv(TABLES / "validate_nan_disagreements.csv")
display(nan_rows[["metapath", "target_position", "gene_set", "n_genes", "z_ref", "z_summary", "null_std_ref"]])

**Behaviour change to accept: zero-variance nulls.** 23 rows disagree on NaN.
In all of them the reference's null standard deviation is <= 1.8e-16: the null
has no variance, and the reference turns rounding residue into a finite z
(|z| <= 1.5). The laptop path reports NaN, as the design's zero-variance rule
says. The first validation run also had one disagreement the other way; the rule
was corrected (decisions.md) and the run repeated.

## The app on a 24 GB laptop

`scripts/verify_null_bundle_app.py` launches the real Streamlit app with the full
bundle, drives it with Playwright, and samples the server's memory.

In [ ]:
display(pd.read_csv(TABLES / "verify_query_timing.csv"))
display(pd.read_csv(TABLES / "verify_app.csv"))

In [ ]:
Image(filename=str(FIGURES / "verify_memory_and_latency.png"))

In [ ]:
Image(filename=str(FIGURES / "verify_app_worked_example.png"))

**Hypothesis against result.**

| expectation | measured | |
|---|---|---|
| peak RSS <= 1.5 GB | 1.19 GB (was 33.6 GB of preload) | met |
| <= 1 s for 18 genes | 1.29 s first query, 0.32 s repeat | first query not met |
| <= 3 s for 200 genes | 1.67 s | met |
| z matches the matrix adapter | see validation | met |

The first 18-gene query misses 1 s because it loads and degree-weights the edge
matrices for all 52 metapaths; later queries reuse them. A page render takes
5.3-5.6 s, most of it the unchanged path enumeration and subgraph drawing.

## Behaviour changes a reviewer must accept

1. **No DWPC matrices at query time.** The app needs `data/null_bundle/` (built on
   Alpine with `hpc/submit_null_bundle.sh`) instead of `data/dwpc_cache/`, and
   refuses a bundle built from different data files.
2. **DWPC residues are zero.** Values with |x| <= 1e-15 (578 entries, all in
   GpBPpGpBP, ~1e-18) are treated as 0.
3. **Zero-variance nulls are NaN** even where the old code produced a
   rounding-driven finite z (table above).
4. **Gene IDs are a set.** A duplicated ID counts once.
5. **`min_stratum_size` is fixed by the bundle** (50); changing it means rebuilding.

## Conclusion

The web query runs on a laptop: 1.19 GB peak instead of a 33.6 GB preload,
about 0.3-1.7 s per query, with z equal to the matrix-based adapter within
7.1e-10 relative across 147,680 validation rows. The same bundle tables are
Postgres-loadable for the production tool.